In [88]:
import torch
from torch import nn
from d2l import torch as d2l

In [89]:
# Sequence-To-Sequence Parameter Initialization

def init_seq2seq(
    module: nn.Module,
) -> None:
    
    if type(module) is nn.Linear:
        nn.init.xavier_uniform_(
            module.weight
        )

    if type(module) is nn.GRU:        
        for name, parameter in module.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(
                    parameter
                )

In [90]:
# Sequence-to-Sequence Encoder

class Seq2SeqEncoder(d2l.Encoder):
    
    def __init__(
        self,
        vocab_size: int,
        embed_size: int,
        num_hiddens: int,
        num_layers: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_size,
        )
        
        self.rnn = d2l.GRU(
            num_inputs=embed_size,
            num_hiddens=num_hiddens,
            num_layers=num_layers,
            dropout=dropout,
        )
        
        self.apply(
            init_seq2seq
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
        *args: object,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,  
    ]:
        
        # [nn.Embedding]:
        # [B, T] -> [T, B] -> [T, B, E]
        embeddings = self.embedding(
            X.T.to(dtype=torch.long)
        )
        
        # [nn.GRU]: 
        # inputs : [T, B, E]
        # outputs: [T, B, H]
        # state  : [L, B, H]
        outputs, state = self.rnn(
            embeddings 
        )
        
        return outputs, state

In [91]:
# Encoder tensor shape 체크

vocab_size = 10
embed_size = 8
num_hiddens = 16
num_layers = 2
batch_size = 4
num_steps = 9

encoder = Seq2SeqEncoder(
    vocab_size=vocab_size,
    embed_size=embed_size,
    num_hiddens=num_hiddens,
    num_layers=num_layers,
)

# [B, T] = [4, 9]
X = torch.zeros(
    (batch_size, num_steps),
    dtype=torch.long,
)

with torch.no_grad():
    encoder_outputs, encoder_state = encoder(X)

print(
    "Input shape:",
    tuple(X.shape),
)
print(
    "Encoder outputs shape:",
    tuple(encoder_outputs.shape),
)
print(
    "Encoder state shape:",
    tuple(encoder_state.shape),
)

d2l.check_shape(
    encoder_outputs,
    (num_steps, batch_size, num_hiddens),
)
d2l.check_shape(
    encoder_state,
    (num_layers, batch_size, num_hiddens),
)

Input shape: (4, 9)
Encoder outputs shape: (9, 4, 16)
Encoder state shape: (2, 4, 16)
